# FaceGuard v2 — Presentation Attack Detection (PAD) MobileNetV3-Small Fine-Tuning

**Architecture:** `MobileNetV3-Small` (RGB Binary Classifier: `1.0` = Bona-fide / Genuine, `0.0` = Presentation Attack / Spoof)  
**Training Strategy:** Two-stage transfer learning with PyTorch DistributedDataParallel (DDP) across 2x T4 GPUs:  
- **Stage 0 (Sanity):** Overfit verification on a balanced mini-cohort before launching full training  
- **Stage 1 (Head-only):** Final linear layer (`classifier[-1]`) trained at `lr=1e-3` with frozen backbone  
- **Stage 2 (Finetune):** Last 3 inverted residual blocks (`features[-3:]`) + classifier at `lr=1e-4 / 1e-3` with frozen BatchNorm  
**Data Splitting:** Subject-disjoint train / validation / test manifests from official CelebA-Spoof JSON labels  
**Target Metrics:** Equal Error Rate (EER), APCER, BPCER, ACER pursuant to ISO/IEC 30107-3  
**Output Artifacts:** `mobilenetv3_pad.onnx` (dynamic batch, opset 17) & `pad_metrics.json`

---
### Kaggle Setup Instructions:
1. **Accelerator:** In the right sidebar, select **GPU T4 x2** (or **GPU P100**).
2. **Internet:** In the right sidebar, ensure **Internet On** is selected.
3. **Attach Inputs:** Click **`+ Add Input`**:
   - Attach **CelebA-Spoof** dataset (e.g. search `celeba spoof` or `marquis03/celeba-spoof-for-face-antispoofing`).
   - Attach **`faceguard-kaggle-source`** (the code bundle built via `scripts/build_kaggle_bundle.py`).

In [ ]:
# --- 1. Environment & Multi-GPU Verification ---
import os, sys, subprocess, shutil, json, zipfile
from pathlib import Path
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
gpu_count = torch.cuda.device_count()
print(f"Detected GPU Count: {gpu_count}")
for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  [GPU {i}] {props.name} ({props.total_memory / (1024**3):.2f} GB VRAM)")

if gpu_count < 1:
    raise RuntimeError("CUDA GPU accelerator is required! Please select GPU T4 x2 in the Kaggle sidebar.")

# Install required runtime packages if missing
try:
    import onnx, onnxruntime
    print(f"[✓] ONNX Runtime: {onnxruntime.__version__}")
except ImportError:
    print("Installing onnx and onnxruntime...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=True)

WORK_ROOT = Path("/kaggle/working/faceguard")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print(f"[✓] Working root initialized: {WORK_ROOT}")


In [ ]:
# --- 2. Locate and Unpack FaceGuard Code Bundle ---
SOURCE_ROOT = None
code_candidates = [
    Path("/kaggle/input/faceguard-kaggle-source"),
    Path("/kaggle/input/datasets/jit007/faceguard-kaggle-source"),
]

# Also scan /kaggle/input for any source zip
if Path("/kaggle/input").exists():
    for root, dirs, files in os.walk("/kaggle/input"):
        if "src" in dirs and "scripts" in dirs:
            SOURCE_ROOT = Path(root)
            break
        for f in files:
            if "faceguard" in f.lower() and f.endswith(".zip"):
                code_candidates.append(Path(root) / f)

target_src = WORK_ROOT / "source"
if SOURCE_ROOT is None:
    for cand in code_candidates:
        if cand.is_dir() and (cand / "src" / "faceguard").is_dir():
            SOURCE_ROOT = cand
            break
        if cand.is_file() and cand.suffix == ".zip":
            print(f"Unpacking source bundle from {cand}...")
            target_src.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(cand, "r") as zf:
                zf.extractall(target_src)
            SOURCE_ROOT = target_src
            break
        if cand.is_dir():
            zips = list(cand.glob("*.zip"))
            if zips:
                print(f"Unpacking source bundle from {zips[0]}...")
                target_src.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(zips[0], "r") as zf:
                    zf.extractall(target_src)
                SOURCE_ROOT = target_src
                break

if SOURCE_ROOT is None or not (SOURCE_ROOT / "src" / "faceguard").is_dir():
    raise RuntimeError("Could not locate faceguard-kaggle-source code bundle! Please attach it under /kaggle/input.")

sys.path.insert(0, str((SOURCE_ROOT / "src").resolve()))
print(f"[✓] Project source active from: {SOURCE_ROOT}")


In [ ]:
# --- 3. Locate CelebA-Spoof Dataset Root ---
def find_celeba_root(start_path: Path) -> Path | None:
    for label_sub in ["metas/intra_test/train_label.json", "metas/labels/train_label.json"]:
        candidates = list(start_path.rglob(label_sub))
        if candidates:
            metas_dir = candidates[0].parent.parent
            return metas_dir.parent
    return None

CELEBA_ROOT = find_celeba_root(Path("/kaggle/input"))

# If CelebA-Spoof is uploaded as a .zip, unpack it
if CELEBA_ROOT is None and Path("/kaggle/input").exists():
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if "celeba" in f.lower() and f.endswith(".zip"):
                unzip_dir = WORK_ROOT / "data" / "celeba_spoof"
                unzip_dir.mkdir(parents=True, exist_ok=True)
                print(f"Unpacking CelebA-Spoof archive from {Path(root) / f}...")
                with zipfile.ZipFile(Path(root) / f, "r") as zf:
                    zf.extractall(unzip_dir)
                CELEBA_ROOT = find_celeba_root(unzip_dir)
                if CELEBA_ROOT:
                    break
        if CELEBA_ROOT: break

if CELEBA_ROOT is None:
    raise RuntimeError("CelebA-Spoof dataset not found! Please attach CelebA-Spoof under /kaggle/input (e.g. marquis03/celeba-spoof-for-face-antispoofing).")

print(f"[✓] CELEBA_ROOT confirmed: {CELEBA_ROOT}")


In [ ]:
# --- 4. Prepare Official Subject-Disjoint PAD Manifests ---
from faceguard.pad_manifest import prepare_celeba_spoof_manifests

MANIFEST_DIR = WORK_ROOT / "pad_manifests"
if MANIFEST_DIR.exists():
    shutil.rmtree(MANIFEST_DIR)

print("Generating verified subject-disjoint train/val/test manifests from official JSON labels...")
manifest_report = prepare_celeba_spoof_manifests(
    dataset_root=CELEBA_ROOT,
    output_dir=MANIFEST_DIR,
    seed=42,
    validation_subject_fraction=0.10,
    train_limit=100_000,
    validation_limit=20_000,
    test_limit=40_000,
)

print("\n" + "="*60)
print("PAD MANIFEST SUMMARY (Subject-Disjoint Splits):")
for split_name in ["train", "val", "test"]:
    s = manifest_report["splits"][split_name]
    print(f"  • {split_name.upper():5s}: {s['samples']:,} samples across {s['subjects']:,} subjects | "
          f"Bona-fide (Live): {s['target_live_counts'].get(1, 0):,}, Attack (Spoof): {s['target_live_counts'].get(0, 0):,}")
print("="*60)


In [ ]:
# --- 5. Distributed 2-GPU MobileNetV3 Training via torchrun ---
PAD_OUTPUT = WORK_ROOT / "pad_output"
PAD_OUTPUT.mkdir(parents=True, exist_ok=True)

nproc = min(2, torch.cuda.device_count())
print(f"Launching PyTorch DDP on {nproc} GPU(s)...")

train_script = SOURCE_ROOT / "scripts" / "train_pad.py"

cmd = [
    sys.executable, "-m", "torch.distributed.run",
    "--standalone",
    f"--nproc_per_node={nproc}",
    str(train_script),
    "--dataset-root", str(CELEBA_ROOT),
    "--manifest-dir", str(MANIFEST_DIR),
    "--output-dir", str(PAD_OUTPUT),
    "--head-epochs", "2",
    "--finetune-epochs", "6",
    "--batch-size-per-gpu", "64",
    "--workers-per-gpu", "2",
    "--early-stopping-patience", "4",
    "--seed", "42",
    "--acknowledge-non-commercial-terms",
]

print(f"Running distributed training command:\n{' '.join(cmd)}\n")
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()

if process.returncode != 0:
    raise RuntimeError(f"PAD training failed with exit code {process.returncode}")
print("\n[✓] Training run completed successfully!")


In [ ]:
# --- 6. Verify Output Artifacts & Metrics ---
metrics_path = PAD_OUTPUT / "pad_metrics.json"
onnx_path = PAD_OUTPUT / "mobilenetv3_pad.onnx"
best_pth_path = PAD_OUTPUT / "mobilenetv3_pad_best.pth"

if not metrics_path.exists() or not onnx_path.exists():
    raise RuntimeError("Training output artifacts are missing!")

with metrics_path.open() as f:
    metrics = json.load(f)

val_m = metrics["validation_threshold"]
test_m = metrics["test"]

print("="*65)
print("🏆 PAD TEST SET EVALUATION (ISO/IEC 30107-3 Standard Metrics):")
print(f"  • Calibrated Threshold (τ* from Val EER): {val_m['threshold']:.4f}")
print(f"  • Validation EER:                         {val_m['acer']*100:.2f}%")
print(f"  • Test APCER (Spoof accepted as Live):    {test_m['apcer']*100:.2f}%")
print(f"  • Test BPCER (Live rejected as Spoof):    {test_m['bpcer']*100:.2f}%")
print(f"  • Test ACER (Average Classification Err): {test_m['acer']*100:.2f}%")
print(f"  • Test ROC-AUC:                           {test_m['auc']:.4f}")
print(f"  • ONNX Parity (Max Abs Diff):             {metrics['onnx_parity_max_abs_difference']:.2e}")
print("="*65)
print(f"[✓] Generated ONNX model: {onnx_path.name} ({onnx_path.stat().st_size / (1024*1024):.2f} MB)")
print(f"[✓] Best PyTorch Weights:  {best_pth_path.name} ({best_pth_path.stat().st_size / (1024*1024):.2f} MB)")


In [ ]:
# --- 7. Download Bundle Helper ---
from IPython.display import FileLink, display

bundle_path = Path("/kaggle/working/pad_model_bundle.zip")
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in [onnx_path, metrics_path, PAD_OUTPUT / "pad_config.json", best_pth_path]:
        if path.exists():
            zf.write(path, arcname=path.name)
            print(f"  [+] Bundled: {path.name}")

print(f"\n🎉 Download Bundle Ready: {bundle_path.name} ({bundle_path.stat().st_size / (1024*1024):.2f} MB)")
print("Click the link below to download your fine-tuned PAD model:")
display(FileLink("pad_model_bundle.zip"))
